# SEBAL Notebook 7: Refined SEBAL ET Workflow    
**Scene:** Landsat 7 (2020-01-19)  
**Region:** Mississippi Delta (EPSG:32615)

This notebook refines the SEBAL energy balance using stability correction to update H, LE, EF, and ET estimates.

## Starting Point — First-Pass SEBAL Results

The first-pass SEBAL energy balance was completed earlier.  
The following rasters already exist in `04_indices` and will be used here:

• H (first-pass sensible heat flux)  
• Rn (net radiation)  
• G (soil heat flux)  
• ETinst (first-pass instantaneous ET)  
• ET24 (first-pass daily ET)

This notebook continues with stability correction and refined flux computation.

## 1. Setup and inport

In [1]:
import os
import rasterio
import numpy as np
from Utility import get_file_dataframe
from Utility import load_rasters
from Utility import calculate_NDVI
from Utility import calculate_albedo
from Utility import extract_mask_mean
from Utility import read_raster
from Utility import write_raster
from Utility import check_raster

## 2. Set SEBAL working directory

In [2]:
os.chdir("..")
os.getcwd()

'G:\\Collaborations\\Mentee\\UF_Anitha Madapakula\\Scripts\\Python\\SEBAL_20200119_MSDelta'

## 3. Define directories and input

In [3]:
indices_dir = "04_indices"
date = "20200119"
hour_tag = "16Z"
region = "MSDelta"

## 4. Load input rasters

In [4]:

H_refined_path = os.path.join(indices_dir, f"H_refined_{date}_{hour_tag}_{region}.tif")
Rn_path        = os.path.join(indices_dir, f"Rn_{date}_{hour_tag}_{region}.tif")
G_path         = os.path.join(indices_dir, f"G_{date}_{hour_tag}_{region}.tif")
ETinst_fp_path     =  os.path.join(indices_dir, f"ETinst_firstpass_{date}_{hour_tag}_{region}.tif")
ET24_fp_path     =  os.path.join(indices_dir, f"ET24_firstpass_{date}_{hour_tag}_{region}.tif")

H_refined, profile, _ = read_raster(H_refined_path)
Rn, _, _         = read_raster(Rn_path)
G, _, _          = read_raster(G_path)
ETinst_fp , _, _      = read_raster(ETinst_fp_path)
ET24_fp, _, _          = read_raster(ET24_fp_path)

## 5. Refined Laten Heat (LE)
### 5.1 Compute refined latent heat flux (LE)

In [5]:
LE_refined = np.full(Rn.shape, -9999.0, dtype="float32")

valid_LE = (Rn > -9999) & (G > -9999) & (H_refined > -9999)
LE_refined[valid_LE] = Rn[valid_LE] - G[valid_LE] - H_refined[valid_LE]

#LE_refined[valid_LE] = np.maximum(LE_refined[valid_LE], 0.0)

print("LE_refined min/max:",
      float(LE_refined[LE_refined > -9999].min()),
      float(LE_refined[LE_refined > -9999].max()))

LE_refined min/max: -800.2109985351562 415.78106689453125


### 5.2 Clamp LE to physical lower bound

In [6]:
LE_refined_clamped = np.full(Rn.shape, -9999.0, dtype="float32")

valid = LE_refined > -9999
LE_refined_clamped[valid] = np.maximum(LE_refined[valid], 0)

print("Clamped LE min/max:",
      float(LE_refined_clamped[valid].min()),
      float(LE_refined_clamped[valid].max()))

Clamped LE min/max: 0.0 415.78106689453125


### 5.3 Saved clamped LE

In [7]:
profile.update(dtype="float32", nodata=-9999.0)

LE_refined_out = np.where(LE_refined_clamped > -9999, LE_refined_clamped, -9999).astype("float32")
LE_refined_path = os.path.join(indices_dir, f"LE_refined_{date}_{hour_tag}_{region}.tif")

write_raster(raster_path=LE_refined_path, profile=profile, value=LE_refined_out)

Saved: 04_indices\LE_refined_20200119_16Z_MSDelta.tif


In [8]:
LE_fp_path = os.path.join(indices_dir, f"LE_firstpass_{date}_{hour_tag}_{region}.tif")
LE_ref_path = os.path.join(indices_dir, f"LE_refined_{date}_{hour_tag}_{region}.tif")

LE_fp, _, _ = read_raster(LE_fp_path)
LE_ref, _, _ = read_raster(LE_ref_path)

valid = (LE_fp > -9999) & (LE_ref > -9999)

print("LE all equal?:", np.array_equal(LE_fp[valid], LE_ref[valid]))
print("LE max abs diff:", float(np.max(np.abs(LE_ref[valid] - LE_fp[valid]))))

LE all equal?: True
LE max abs diff: 0.0


## 6. Refined evaporative fraction (EF)

In [9]:
EF_final = np.full(Rn.shape, -9999.0, dtype="float32")

AE = np.full(Rn.shape, -9999.0, dtype="float32")
valid_AE = (Rn > -9999) & (G > -9999)
AE[valid_AE] = Rn[valid_AE] - G[valid_AE]

valid_EF = (LE_refined_clamped > -9999) & (AE > 0)

EF_final[valid_EF] = LE_refined_clamped[valid_EF] / AE[valid_EF]

print("EF_final min/max:",
      float(EF_final[EF_final > -9999].min()),
      float(EF_final[EF_final > -9999].max()))
profile.update(dtype="float32", nodata=-9999.0)

EF_out = np.where(EF_final > -9999, EF_final, -9999).astype("float32")
EF_path = os.path.join(indices_dir, f"EF_refined_{date}_{hour_tag}_{region}.tif")

write_raster(raster_path=EF_path, profile=profile, value=EF_out)

EF_final min/max: 0.0 1.0
Saved: 04_indices\EF_refined_20200119_16Z_MSDelta.tif


## 7. Refined instantaneous ET

In [10]:
lambda_v = 2.45e6

ETeq = np.full(Rn.shape, -9999.0, dtype="float32")

denom = Rn - G
valid_ETeq = (Rn > -9999) & (G > -9999) & (denom > 0)

ETeq[valid_ETeq] = denom[valid_ETeq] * 3600 / lambda_v

ETinst_final = np.full(Rn.shape, -9999.0, dtype="float32")

valid_ETinst = (EF_final > -9999) & (ETeq > -9999)

ETinst_final[valid_ETinst] = EF_final[valid_ETinst] * ETeq[valid_ETinst]

print("ET_inst min/max:",
      float(ETinst_final[ETinst_final > -9999].min()),
      float(ETinst_final[ETinst_final > -9999].max()))

# Update raster profile settings for saving
profile.update(dtype="float32", nodata=-9999.0)

# Replace nodata and convert format
ETinst_out = np.where(ETinst_final > -9999, ETinst_final, -9999).astype("float32")

# Output file path for refined ETinst raster
ETinst_path = os.path.join(indices_dir, f"ETinst_refined_{date}_{hour_tag}_{region}.tif")

# Save raster
write_raster(raster_path=ETinst_path, profile=profile, value=ETinst_out)

ET_inst min/max: 0.0 0.610943615436554
Saved: 04_indices\ETinst_refined_20200119_16Z_MSDelta.tif


## 8. Daily ET from instantaneous ET

In [11]:
ET24_refined = np.full(ETinst_final.shape, -9999.0, dtype="float32")

valid_ET24 = (ET24_fp  > -9999) & (ETinst_final > -9999) & (ETinst_fp > 0)

ET24_refined[valid_ET24] = (
    ETinst_final[valid_ET24]
    * ETinst_final[valid_ET24]
    / ETinst_fp[valid_ET24]
)

print("ET24_refined min/max:",
      float(ET24_refined[ET24_refined > -9999].min()),
      float(ET24_refined[ET24_refined > -9999].max()))

ET24_out = np.where(ET24_refined > -9999, ET24_refined, -9999).astype("float32")
ET24_path = os.path.join(indices_dir, f"ET24_refined_{date}_{hour_tag}_{region}.tif")

write_raster(raster_path=ET24_path, profile=profile, value=ET24_out)

ET24_refined min/max: 8.968431330913518e-08 0.610943615436554
Saved: 04_indices\ET24_refined_20200119_16Z_MSDelta.tif
